# How Does Global AI Adoption Rate Compare with Productivity Change? — Phase II
### ENGE707: Data Engineering and Machine Learning Assignment — LCS-Based Prediction

Suemon Kwok (14883335), Quentin Masoe (23191960), Avathanshu Bhat (23227963)

**Dataset:** Global AI Adoption and Workforce Impact Dataset (`ai_company_adoption.csv`)

This notebook continues on from the Phase I scoping report and works through the Phase II tasks (LCS baseline, preprocessing, improved LCS system, experiments, model comparison, interpretation, and discussion).

In [1]:
import os
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

## Task 1 - Project Phase I Summary

### 1.1 Problem restatement

- **Problem:** predict a company's reported productivity change from its AI adoption behaviour.
- **Target variable:** `productivity_change_percent` (continuous, %) — this is a **regression** problem.
- **Predictor (Phase I scope):** `ai_adoption_rate` (% of operations/processes using AI).
- **Prediction objective:** given a company-quarter's AI adoption rate (and, from Phase II onward, other available features), estimate the productivity change the company is likely to report.

In [ ]:
# Load the dataset (extract from archive.zip first if the CSV isn't already alongside this notebook)
CSV_NAME = "../ai_company_adoption.csv"
ZIP_NAME = "archive.zip"

if not os.path.exists(CSV_NAME) and os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")

df = pd.read_csv(CSV_NAME)
df.shape

FileNotFoundError: [Errno 2] No such file or directory: 'ai_company_adoption.csv'

### 1.2 Dataset source, size, and characteristics

In [ ]:
# Quick structural recap carried over from Phase I
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Distinct companies: {df['company_id'].nunique():,}")
print(f"Countries: {df['country'].nunique()}")
print(f"Industries: {df['industry'].nunique()}")
print(f"Survey period: {df['survey_year'].min()}-{df['quarter'].min()} to {df['survey_year'].max()}-{df['quarter'].max()}")

Rows: 150,000
Columns: 43
Distinct companies: 10,000
Countries: 30
Industries: 9
Survey period: 2023-Q1 to 2026-Q4


In [ ]:
# Dtype breakdown
df.dtypes.value_counts()

float64    15
int64      14
str        14
Name: count, dtype: int64

- **Source:** Global AI Adoption and Workforce Impact Dataset (Kaggle, CC0 licence).
- **Size:** 150,000 company-quarter survey responses from 10,000 distinct companies, across 30 countries and 9 industries, surveyed quarterly Q1 2023 – Q4 2026.
- **Features:** 43 columns total — a mix of numeric (AI investment, budget, maturity, satisfaction, financial metrics), categorical (industry, country, company size, tool used), and identifier columns.
- **Phase I focus variables:** `ai_adoption_rate` and `productivity_change_percent` only, per the narrowed scope agreed after Phase I lecturer feedback.

### 1.3 Main data-quality issues identified in Phase I

In [ ]:
# Re-confirm the Phase I data-quality findings for the two focus variables
print("Missing values:")
print(df[["ai_adoption_rate", "productivity_change_percent"]].isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate response_id:", df["response_id"].duplicated().sum())

print("\nai_adoption_rate range:", df["ai_adoption_rate"].min(), "-", df["ai_adoption_rate"].max())
print("productivity_change_percent range:", df["productivity_change_percent"].min(), "-", df["productivity_change_percent"].max())

Missing values:
ai_adoption_rate               0
productivity_change_percent    0
dtype: int64



Duplicate rows: 0
Duplicate response_id: 0

ai_adoption_rate range: 0.0 - 100.0
productivity_change_percent range: 0.0 - 34.36


- No missing values and no duplicate records were found in either focus variable.
- Both variables sit within their valid documented ranges (0–100% and 0–34.36% respectively), so no impossible values needed correcting.
- IQR-based outlier checks flagged well under 1% of records in each variable (504 and 529 respectively) — these were treated as genuine high-adoption/high-productivity companies, not data-entry errors.
- The dataset is an **unbalanced panel**: 10,000 companies each contribute an unequal number of quarterly rows, so the 150,000 records are not fully independent observations.
- A secondary rise near 0–2% in the productivity distribution suggested a distinct "zero/low-productivity" subgroup, later linked to lower mean AI adoption (18% vs. 36% overall).

### 1.4 Why this problem is suitable for an LCS-based solution

- The target (`productivity_change_percent`) is a **continuous outcome driven by conditions across several company attributes** (adoption behaviour, workforce factors, financial context) — LCS systems are well suited to problems where the relationship is expressed as a set of localised if-then rules over feature conditions, rather than a single global function.
- The dataset is **tabular with mixed numeric and categorical features**, which matches the input format LCS implementations (e.g. eLCS, XCS) are designed for.
- Phase I already showed the AI-adoption/productivity relationship is **not purely linear or global** — the zero-productivity subgroup behaved differently from the rest of the sample. LCS's rule-based, niche-covering approach can capture this kind of subgroup structure explicitly, and its rules remain human-readable, which supports the interpretability requirement in Task 7.
- The moderate size of the dataset (150,000 rows, reducible to a manageable feature set) is within the range LCS implementations such as eLCS can realistically train on.

### Note on Task 2 scope (resolved)

Confirmed direction: keep the Phase I two-variable scope (`ai_adoption_rate` -> `productivity_change_percent`) for the **original LCS baseline** in Task 2, since the other 41 columns were already dropped during the Phase I cleaning step and never carried forward. Any feature expansion, if used, will be introduced and justified as part of the **improved system** in Tasks 3/4, not the baseline.

## Task 2 - Original LCS System on Raw Dataset

### 2.1 LCS variant selection

**eLCS**, using the project's trimmed single-file implementation `elcs_custom.py` (derived from the `scikit-eLCS` / `skeLCS` eLCS code: same algorithm, with timers, tracking, reboot/pickling, debug printing and roulette selection removed). It keeps the scikit-learn `fit` / `predict` / `score` interface.

In [ ]:
import itertools

# Custom, self-contained eLCS (trimmed from skeLCS) + project settings + Decision Tree factory.
# Keep elcs_custom.py in the same folder as this notebook.
from elcs_custom import (
    eLCS, make_elcs, make_decision_tree,
    N_SEEDS, TRAIN_SUBSAMPLE_SIZE, TEST_EVAL_SIZE, POPULATION_SIZE_N, LEARNING_ITERATIONS, MAX_DEPTH,
)

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    multilabel_confusion_matrix,
)

### 2.2 Minimal processing required for LCS compatibility

`skeLCS.eLCS` requires:
- all feature values to be numeric (or `NaN`),
- the target (`phenotype`) to be numeric **and discrete** — this implementation always treats the phenotype as a classification target, so a continuous variable cannot be passed directly.

`ai_adoption_rate` is already numeric, so no change is needed there. `productivity_change_percent`, however, is continuous, so it is minimally processed by binning it into three quantile-based classes (Low / Medium / High) and label-encoding them as integers 0/1/2. This is the smallest possible change that makes the existing Phase I target compatible with the LCS code — no algorithmic modification is made to eLCS itself.

In [ ]:
# Minimal processing: discretise the continuous target into 3 balanced classes so it is
# compatible with eLCS's classification-only phenotype requirement
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], q=3, labels=[0, 1, 2]
).astype(int)

df["productivity_class"].value_counts().sort_index()

productivity_class
0    50023
1    50031
2    49946
Name: count, dtype: int64

In [ ]:
# Baseline feature set: the single Phase I predictor, kept unmodified
X = df[["ai_adoption_rate"]].values.astype(float)
y = df["productivity_class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((120000, 1), (30000, 1))

### 2.3 LCS parameters

Default `skeLCS.eLCS` hyperparameters are used for the baseline (no tuning at this stage — tuning is reserved for the improved system in Task 4):

| Parameter | Value | Meaning |
|---|---|---|
| `learning_iterations` | 10,000 | number of training iterations (rule-learning cycles) |
| `N` | 1,000 | maximum micro-classifier population size |
| `nu` | 5 | fitness pressure exponent |
| `chi` | 0.8 | crossover probability |
| `mu` | 0.04 | mutation probability |
| `p_spec` | 0.5 | probability of specifying an attribute on covering |
| `discrete_attribute_limit` | 10 | max unique values before an attribute is treated as continuous |
| `random_state` | 42 | fixed for reproducibility |

In [ ]:
# Train the original, unmodified eLCS baseline
elcs_baseline = eLCS(learning_iterations=10000, random_state=42)
elcs_baseline.fit(X_train, y_train)
print("eLCS baseline trained.")

eLCS baseline trained.


In [ ]:
# Baseline performance (eLCS prediction is pure-Python and slow at scale, so train
# accuracy is estimated on a random subsample rather than all 120,000 training rows)
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_train), size=5000, replace=False)

train_acc = elcs_baseline.score(X_train[sample_idx], y_train[sample_idx])
y_pred_test = elcs_baseline.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
coverage = elcs_baseline.get_final_instance_coverage()

print(f"Train accuracy (5,000-row sample): {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Final instance coverage: {coverage:.4f}")

Train accuracy (5,000-row sample): 0.5387
Test accuracy: 0.5397
Final instance coverage: 1.0000


In [ ]:
print(classification_report(y_test, y_pred_test, target_names=["Low", "Medium", "High"]))

              precision    recall  f1-score   support

         Low       0.61      0.72      0.66     10005
      Medium       0.00      0.00      0.00     10006
        High       0.50      0.90      0.64      9989

    accuracy                           0.54     30000
   macro avg       0.37      0.54      0.43     30000
weighted avg       0.37      0.54      0.43     30000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### 2.4 Baseline results summary

Random-guess accuracy for 3 balanced classes is ~33%. The single-feature eLCS baseline should be compared against this floor once results are in — a low margin above chance would support broadening the feature set (or improving preprocessing) as the Task 3/4 improvement, without changing the Task 2 baseline scope itself.

## Task 3 - Data Preprocessing and Feature Engineering

In [ ]:
# Target: 3-class quantile binning (Low/Medium/High). Identical for every model so comparisons are fair.
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], q=3, labels=[0, 1, 2]
).astype(int)
print(df["productivity_class"].value_counts())

**Feature set and encoding (7 source columns -> 15 model columns).** Only predictors available *before* the outcome are used (no leakage columns such as `jobs_displaced` / `revenue_growth_percent`). `ai_adoption_stage` is **ordinal** (none < pilot < partial < full), so it is mapped to 0/1/2/3 in that order; `industry` is **nominal**, so it is one-hot encoded into 9 columns. Result: 5 numeric + 1 ordinal + 9 one-hot = 15 columns.

In [ ]:
SELECTED_FEATURE_COLUMNS = [
    "ai_adoption_rate", "ai_maturity_score", "ai_failure_rate",
    "ai_training_hours", "task_automation_rate", "ai_adoption_stage", "industry",
]
X_df = df[SELECTED_FEATURE_COLUMNS].copy()

STAGE_ORDER = {"none": 0, "pilot": 1, "partial": 2, "full": 3}
X_df["ai_adoption_stage"] = X_df["ai_adoption_stage"].map(STAGE_ORDER)
X_df = pd.get_dummies(X_df, columns=["industry"], prefix="industry", dtype=float)

FEATURE_NAMES = list(X_df.columns)
X_fe = X_df.values.astype(float)                 # eLCS feature matrix (150000, 15)
y_fe = df["productivity_class"].values.astype(int)
print("X shape:", X_fe.shape, "| y shape:", y_fe.shape)

No scaling was applied: eLCS matches continuous attributes by range internally, and the Decision Tree used later in the model comparison is scale-invariant, so scaling would add a processing step without changing either model's behaviour.

## Task 4 - Improved LCS-Based System

The improved system is eLCS with **correct-set subsumption** switched on (rules merge into more general ones after each correct-set update), run on the preprocessed 15-feature data. All other parameters match the original eLCS run so the only change is the subsumption setting.

### Shared evaluation protocol (Tasks 4-5)
- Seeds 0-9: a fresh 80/20 stratified split per seed, so each of the 10 trials is a genuinely different split/run (not one run repeated).
- eLCS: 1,000-row training subsample per seed, `N` = 1.5 x 1,000, `learning_iterations` = 5 x 1,000, `random_state` = seed.
- Decision Tree: trained on the full training partition per seed, `random_state` = seed.
- Every model is scored on the same 5,000-row test slice per seed, so accuracies are paired by seed.

In [ ]:
# Project settings come from elcs_custom.py
CLASS_NAMES = ["Low", "Medium", "High"]
print(f"Population size N = {POPULATION_SIZE_N}, learning_iterations = {LEARNING_ITERATIONS}")

def run_seeds(label, make_model, X, y, train_subsample=None):
    """Fit a fresh model per seed (0..N_SEEDS-1) on a fresh stratified 80/20 split; score on the fixed test slice."""
    accs, y_true_all, y_pred_all, last = [], [], [], None
    for seed in range(N_SEEDS):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y
        )
        if train_subsample is not None:
            idx_sub = np.random.RandomState(seed).choice(len(X_tr), train_subsample, replace=False)
            X_tr, y_tr = X_tr[idx_sub], y_tr[idx_sub]
        X_ev, y_ev = X_te[:TEST_EVAL_SIZE], y_te[:TEST_EVAL_SIZE]

        model = make_model(seed)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_ev)
        acc = accuracy_score(y_ev, y_pred)
        accs.append(acc)
        y_true_all.extend(y_ev.tolist()); y_pred_all.extend(y_pred.tolist())
        last = model
        print(f"Seed {seed}: accuracy = {acc:.4f}")
    return dict(label=label, accs=accs, y_true=np.array(y_true_all), y_pred=np.array(y_pred_all), last_model=last)

def report(res):
    """Mean/std, pooled confusion matrix, classification report, per-class TP/TN/FP/FN."""
    print(f"\n{res['label']} -- mean accuracy over {N_SEEDS} seeds: "
          f"{np.mean(res['accs']):.4f} (std {np.std(res['accs']):.4f})")
    print("\nPooled confusion matrix (rows = actual, columns = predicted):")
    print(confusion_matrix(res["y_true"], res["y_pred"]))
    print("\nClassification report (pooled across all seeds):")
    print(classification_report(res["y_true"], res["y_pred"], target_names=CLASS_NAMES, zero_division=0))
    print("Per-class TP / TN / FP / FN (one-vs-rest):")
    for name, m in zip(CLASS_NAMES, multilabel_confusion_matrix(res["y_true"], res["y_pred"])):
        tn, fp, fn, tp = m.ravel()
        print(f"  {name:7s}: TP={tp:5d}  TN={tn:5d}  FP={fp:5d}  FN={fn:5d}")

def export_rules(res, filename):
    """Export the last seed's eLCS rule population and return it as a DataFrame."""
    res["last_model"].export_final_rule_population(
        headerNames=np.array(FEATURE_NAMES), className="productivity_class",
        filename=filename, DCAL=True,
    )
    rules = pd.read_csv(filename)
    print(f"\nExported {len(rules)} rules to {filename}")
    print(rules.sort_values("Numerosity", ascending=False).head(5))
    return rules

def save_results(res, prefix, n_features, n_rules=None):
    """Write <prefix>_seed_results.csv (seed, model, accuracy) + <prefix>_results.csv summary."""
    pd.DataFrame({"seed": range(N_SEEDS), "model": res["label"], "accuracy": res["accs"]}
                 ).to_csv(f"{prefix}_seed_results.csv", index=False)
    pd.DataFrame([{"model": res["label"], "mean_accuracy": np.mean(res["accs"]),
                   "std_accuracy": np.std(res["accs"]), "n_seeds": N_SEEDS,
                   "n_features": n_features, "n_rules_last_seed": n_rules}]
                 ).to_csv(f"{prefix}_results.csv", index=False)
    print(f"Saved {prefix}_seed_results.csv and {prefix}_results.csv")

### Improved eLCS: 10-seed run

In [ ]:
res_improved = run_seeds(
    "Improved eLCS (+ correct-set subsumption, 15 features)",
    lambda seed: make_elcs(seed, subsumption=True),   # subsumption = the change vs. the original eLCS
    X_fe, y_fe, train_subsample=TRAIN_SUBSAMPLE_SIZE,
)
report(res_improved)

In [ ]:
rules_improved = export_rules(res_improved, "task4b_elcs_subsumption_avathanshu_rules.csv")
save_results(res_improved, "task4b_elcs_subsumption_avathanshu", X_fe.shape[1], len(rules_improved))

## Task 5 - Experiments

### 5.1 Validation strategy

Repeated hold-out over **10 seeds (0-9)**: each seed uses its own 80/20 stratified split (and its own model seed), giving 10 independent trials per model rather than one run repeated. The same split and the same 5,000-row test slice are used for every model within a seed, so accuracies are paired by seed for the significance tests in 5.4.

### 5.2 Run the remaining models (10 seeds each)

In [ ]:
# Original, unmodified eLCS on the preprocessed 15-feature data (same protocol as the improved run)
res_original = run_seeds(
    "Original eLCS (preprocessed, 15 features)",
    lambda seed: make_elcs(seed),
    X_fe, y_fe, train_subsample=TRAIN_SUBSAMPLE_SIZE,
)
report(res_original)
rules_original = export_rules(res_original, "task3_elcs_preprocessed_avathanshu_rules.csv")
save_results(res_original, "task3_elcs_preprocessed_avathanshu", X_fe.shape[1], len(rules_original))

In [ ]:
# Decision Tree: same 7 source columns, integer-coded categoricals (trees only need splittable values),
# trained on the FULL training partition per seed with random_state = seed.
X_dt_df = df[SELECTED_FEATURE_COLUMNS].copy()
for col in ["ai_adoption_stage", "industry"]:
    X_dt_df[col] = LabelEncoder().fit_transform(X_dt_df[col].astype(str))
X_dt = X_dt_df.values.astype(float)

res_dt = run_seeds(
    "Decision Tree (7 selected features)",
    make_decision_tree,
    X_dt, y_fe, train_subsample=None,
)
report(res_dt)

importances = pd.DataFrame({
    "feature": SELECTED_FEATURE_COLUMNS,
    "importance": res_dt["last_model"].feature_importances_,
}).sort_values("importance", ascending=False)
print("\nFeature importances (final seed's tree):")
print(importances.to_string(index=False))
importances.to_csv("task6_decision_tree_feature_importances.csv", index=False)
save_results(res_dt, "task6_decision_tree", X_dt.shape[1])

### 5.3 Evaluation summary

In [ ]:
all_results = [res_original, res_improved, res_dt]
wide = pd.DataFrame({r["label"]: r["accs"] for r in all_results}, index=pd.Index(range(N_SEEDS), name="seed"))
summary = wide.agg(["mean", "std"]).T.sort_values("mean", ascending=False)
summary.to_csv("task5_summary.csv")
summary.round(4)

### 5.4 Statistical significance testing

Every model is scored on the same test rows for each seed, so accuracies are **paired by seed**. Tests used: paired t-test (assumes roughly normal differences) and the Wilcoxon signed-rank test (non-parametric; with n=10 the smallest possible two-sided p is ~0.002). Holm step-down correction is applied across all pairwise comparisons.

In [ ]:
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj

rows = []
for a, b in itertools.combinations(wide.columns, 2):
    d = wide[a] - wide[b]
    t_p = stats.ttest_rel(wide[a], wide[b]).pvalue
    try:
        w_p = stats.wilcoxon(wide[a], wide[b]).pvalue
    except ValueError:
        w_p = np.nan
    rows.append(dict(model_a=a, model_b=b, mean_diff=d.mean(), ttest_p=t_p, wilcoxon_p=w_p))
pairwise = pd.DataFrame(rows)
pairwise["ttest_p_holm"] = holm(pairwise["ttest_p"])
pairwise["wilcoxon_p_holm"] = holm(pairwise["wilcoxon_p"].fillna(1))
pairwise["significant_holm_0.05"] = pairwise["ttest_p_holm"] < 0.05
pairwise.to_csv("task5_pairwise_tests.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
print(pairwise.round(5).to_string(index=False))